In [1]:
import numpy as np

import warnings
from pathlib import Path
import pickle


# from notebooks.data_synchronization_barcode import tsync_barcode
%matplotlib widget

import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import scipy.signal
import pynapple as nap

import pandas as pd
import scipy.io as spio
import os
import tifffile
from totalsync_utils import decode_b64_files
import seaborn as sns
custom_params = {"axes.spines.right": False, "axes.spines.top": False, "figure.figsize": (8, 4)}
sns.set_context("paper")
sns.set_theme(style="ticks", palette="colorblind", font_scale=1.3, rc=custom_params)

from batch2p.extractors.suite3d import create_synced_outputs
import totalsync_2p.sync as sync

In [2]:
root_path = Path("/data/ofl_2p/20251118")
tif_files = [ root_path / "00001/477116_20251118_00001.tif"]
stats_file = root_path / "results/behavior_sync" / "20251118-152602_925_behavior_sync_stats.pkl"
frames_file = root_path / "results/behavior_sync" / "20251118-152602_925_frames_time_idx.npz"
output_dir = root_path / "results/tests"
pin_sheet_file = "/home/battaglia/src/ofl_2p_analysis/docs/pinSheet_2026.json"
b64_file = root_path / "20251118-152602_925.b64"
roi_dir = Path("/data/ofl_2p/20251118/preprocessing_results/00001split/s3d-results-00001split")


In [3]:
with open(stats_file, "rb") as f:
    stats = pickle.load(f)

frames_time_idx = nap.load_file(frames_file)

FileNotFoundError: [Errno 2] No such file or directory: '/data/ofl_2p/20251118/results/behavior_sync/20251118-152602_925_behavior_sync_stats.pkl'

In [4]:
stats = sync.synchronize(tif_files[0], b64_file, output_dir, pin_sheet_file, fill_gaps=True)

/data/ofl_2p/20251118/20251118-152602_925.b64
1383156 packets, ~23.05 minutes


100%|██████████| 1383156/1383156 [00:43<00:00, 31563.69it/s]
/home/battaglia/src/ofl_2p_analysis/totalsync_2p/sync.py:211: UserWarning: There are gaps in the timestamps: [6.68232993e+08 1.38604699e+09]
  warnings.warn(f"There are gaps in the timestamps: {stats['gap_locations']}")
100%|██████████| 40756/40756 [00:04<00:00, 9468.40it/s] 
/home/battaglia/src/ofl_2p_analysis/totalsync_2p/sync.py:260: UserWarning: No barcode detected in the tif file, frame/time alignment will be done by assuming that the first Scanner frame clock pulse corresponds to the first tif frame
  warnings.warn(


In [5]:
stats['frames_time_idx']

Time (s)
-----------  -----
29.087993        0
29.120993        1
29.153993        2
29.187993        3
29.220993        4
29.254993        5
29.287993        6
...
1389.921993  40751
1389.954993  40752
1389.988993  40753
1390.021993  40754
1390.055993  40755
1390.088993  40756
1390.121993  40757
dtype: int64, shape: (40758,)

In [6]:
stats["gap_locations"]

array([], dtype=float64)

def synchronize(tif_file: str, b64_file: str, output_dir: str, pin_sheet_file: str) -> dict:

create_synced_outputs(
    tif_files: list[Path],
    sync_results: list[dict],
    rois_dir: Path,
    behavior_sync_dir: Path,

In [7]:
create_synced_outputs(tif_files, [stats], roi_dir, output_dir, block_size=3)

    Saved F_sync.npz ((13585, 727))
    Saved Fneu_sync.npz ((13585, 727))
    Saved spks_sync.npz ((13585, 727))


In [8]:
stats

{'session': '20251118-152602_925',
 'max_ts_gap': 0,
 'gap_locations': array([], dtype=float64),
 'has_barcode': False,
 'orig_gap_locations': array([6.68232993e+08, 1.38604699e+09]),
 'orig_max_ts_gap': 121000,
 'frames_time_idx': Time (s)
 -----------  -----
 29.087993        0
 29.120993        1
 29.153993        2
 29.187993        3
 29.220993        4
 29.254993        5
 29.287993        6
 ...
 1389.921993  40751
 1389.954993  40752
 1389.988993  40753
 1390.021993  40754
 1390.055993  40755
 1390.088993  40756
 1390.121993  40757
 dtype: int64, shape: (40758,),
 'time_offsets': 0}

In [ ]:
tif = tifffile.TiffFile(tif_files[0])
len(tif.pages)

